# Machine Learning-Based Career Recommendation & Skill Gap Analysis System
### B.Sc. Information Technology Academic Project

This notebook walks through the complete Data Science and Machine Learning workflow:
1. **Exploratory Data Analysis (EDA)**
2. **Feature Engineering & Preprocessing**
3. **Model Training & Multi-Classifier Benchmarking** (Logistic Regression, Decision Tree, Random Forest, KNN, SVM)
4. **Model Evaluation & Confusion Matrix Analysis**
5. **Artifact Serialization with Joblib**

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, os.path.abspath('../src'))

from data_generator import generate_student_dataset, TECHNICAL_SKILLS, SOFT_SKILLS, INTERESTS, CAREERS
from preprocessing import CareerDataPreprocessor
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

print('Libraries imported successfully.')

## 1. Load Dataset & Inspect Schema

In [2]:
raw_path = '../data/raw/student_career_dataset.csv'
if not os.path.exists(raw_path):
    df = generate_student_dataset(n_samples=1600, random_state=42, output_path=raw_path)
else:
    df = pd.read_csv(raw_path)

print(f'Total Records: {df.shape[0]}, Total Features: {df.shape[1]}')
df.head()

## 2. Statistical Summary & Target Class Distribution

In [3]:
print('Target Distribution:')
print(df['career_label'].value_counts())

print('\nSummary Statistics:')
display(df.describe())

## 3. Data Preprocessing & Leakage-Free Train/Test Split

In [4]:
X = df.drop(columns=['student_id', 'career_label'])
y = df['career_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

preprocessor = CareerDataPreprocessor()
preprocessor.fit(X_train, y_train)

X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

print(f'Transformed Training Shape: {X_train_trans.shape}')
print(f'Transformed Test Shape: {X_test_trans.shape}')

## 4. Multi-Model Training & Benchmark Comparison

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'Support Vector Machine': SVC(kernel='rbf', C=1.5, probability=True, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_trans, y_train)
    y_pred = model.predict(X_test_trans)
    
    acc = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')
    
    results.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Weighted F1 (%)': round(f1_w * 100, 2),
        'Macro F1 (%)': round(f1_m * 100, 2)
    })

results_df = pd.DataFrame(results).sort_values(by='Weighted F1 (%)', ascending=False)
display(results_df)

## 5. Classification Report & Confusion Matrix of Best Model

In [6]:
best_rf = models['Random Forest']
y_pred_rf = best_rf.predict(X_test_trans)

print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred_rf))

cm = confusion_matrix(y_test, y_pred_rf, labels=sorted(CAREERS))
print('Confusion Matrix Shape:', cm.shape)